# Inscrição e circunscrição: esfera e poliedros

**Módulo 06 — Geometria Espacial** · Notebook 15 de 18

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/06_geometria_espacial/0615a_inscricao_e_circunscricao_esfera_e_poliedros.ipynb)

**Pré-requisitos**
- Módulo 06 — Geometria Espacial: [`0607a_poliedros_convexos.ipynb`](0607a_poliedros_convexos.ipynb) (poliedros regulares e duais), [`0608a_prismas_e_paralelepipedos.ipynb`](0608a_prismas_e_paralelepipedos.ipynb) (diagonais do cubo), [`0610a_piramide.ipynb`](0610a_piramide.ipynb) (relações métricas e tetraedro regular) e [`0613a_esfera.ipynb`](0613a_esfera.ipynb) (plano tangente, volume e área da esfera)

**Objetivos de aprendizagem** — ao final deste notebook você será capaz de:
- distinguir esfera inscrita, esfera tangente às arestas e esfera circunscrita a um poliedro;
- calcular os três raios para o cubo, o octaedro e o tetraedro regulares;
- usar a relação $r = \frac{3 \cdot \text{volume}}{S}$ para o raio da esfera inscrita;
- comparar os raios das esferas dos cinco poliedros regulares.

**Tempo estimado:** 70 a 90 minutos

### Antes de começar

Rode a célula abaixo antes de qualquer outra. Ela carrega o `utils_3d.py`, o arquivo com as funções de desenho deste módulo.

In [1]:
# Célula de preparação: deixa o notebook pronto para rodar no seu computador ou no Colab.
import sys

if "google.colab" in sys.modules:
    # No Colab, o notebook chega sozinho; baixamos o arquivo de desenhos do repositório.
    import urllib.request
    url = "https://raw.githubusercontent.com/leandrofreiredealmeida/curso_matematica_python/main/06_geometria_espacial/utils_3d.py"
    urllib.request.urlretrieve(url, "utils_3d.py")

import utils_3d as u3d
from ipywidgets import interact
import ipywidgets as widgets
import math

## Quanto ar você compra junto com a bola?

Uma loja vende bolas dentro de caixas cúbicas, do tamanho exato da bola: a bola encosta nas seis faces da caixa. Quanto do espaço da caixa é **ar**?

E o contrário: um brinquedo vem dentro de uma bola de plástico transparente, e o brinquedo é um cubo que encosta na bola pelos oito cantos. Quanto do espaço da bola é **ar**?

🕹️ Faça uma aposta para cada caso e depois confira no widget.

In [ ]:
def comparar_embalagens(caso: str, aresta: float) -> None:
    """Desenha o cubo e a esfera de cada embalagem e calcula quanto do espaço é ar."""
    volume_cubo = aresta**3
    if caso == "Bola dentro da caixa":
        raio = aresta / 2                       # a bola encosta nas seis faces
        volume_esfera = 4 / 3 * math.pi * raio**3
        ar = volume_cubo - volume_esfera
        porcentagem_de_ar = ar / volume_cubo * 100
        opacidade_cubo, opacidade_esfera = 0.2, 0.6
    else:
        raio = aresta * math.sqrt(3) / 2        # metade da diagonal do cubo
        volume_esfera = 4 / 3 * math.pi * raio**3
        ar = volume_esfera - volume_cubo
        porcentagem_de_ar = ar / volume_esfera * 100
        opacidade_cubo, opacidade_esfera = 0.6, 0.2

    # O cubo e a esfera têm o mesmo centro.
    cubo = u3d.poliedro_de_platao("cubo", aresta)
    fig = u3d.criar_figura(titulo=caso)
    fig = u3d.desenhar_poliedro(fig, *cubo, opacidade=opacidade_cubo)
    fig = u3d.desenhar_esfera(
        fig, (0, 0, 0), raio, cor=u3d.CORES["plano"], opacidade=opacidade_esfera
    )
    u3d.mostrar(fig)

    print(f"Aresta do cubo: {aresta:g} cm")
    print(f"Raio da esfera: {raio:.2f} cm")
    print(f"Volume do cubo: {volume_cubo:.1f} cm³")
    print(f"Volume da esfera: {volume_esfera:.1f} cm³")
    print(f"Porcentagem de ar: {porcentagem_de_ar:.1f}%")
    print("Mude a aresta: a porcentagem de ar NÃO depende do tamanho da embalagem.")


interact(
    comparar_embalagens,
    caso=widgets.RadioButtons(
        options=["Bola dentro da caixa", "Cubo dentro da bola"],
        value="Bola dentro da caixa",
    ),
    aresta=widgets.FloatSlider(
        min=2, max=20, step=2, value=20, description="aresta (cm)",
        continuous_update=False,
    ),
);

interactive(children=(RadioButtons(description='caso', options=('Bola dentro da caixa', 'Cubo dentro da bola')…

Na caixa com a bola, quase **48%** do espaço é ar. Na bola com o cubo, mais de **63%**! E esses números não dependem do tamanho: valem para qualquer bola e qualquer caixa nessas posições.

Para chegar a eles, foi preciso saber o raio da esfera **inscrita** no cubo (metade da aresta) e o da esfera **circunscrita** ao cubo (metade da diagonal). Neste notebook você vai aprender a encontrar esses raios para o cubo, o octaedro, o tetraedro e os outros poliedros regulares.

## 1. Esferas inscrita, circunscrita e tangente às arestas

> 📌 **Definição.** Dado um poliedro:
> - uma esfera está **inscrita** no poliedro quando é **tangente a todas as faces** (ela fica dentro do poliedro, encostando em cada face num único ponto). Dizemos também que o poliedro está **circunscrito** à esfera. Raio: $r$.
> - uma esfera está **circunscrita** ao poliedro quando **passa por todos os vértices** (o poliedro fica dentro dela). Dizemos também que o poliedro está **inscrito** na esfera. Raio: $R$.
> - uma esfera é **tangente às arestas** quando encosta em todas as arestas, cada uma num único ponto. Raio: $\rho$.

> ⚠️ **Erro comum.** Inverter as palavras. "Esfera **inscrita** no cubo" quer dizer que a **esfera** fica dentro; "cubo **inscrito** na esfera" quer dizer que o **cubo** fica dentro. Quem está "inscrito" é sempre quem fica **dentro**.

**Nem todo poliedro tem essas esferas.** O centro da esfera circunscrita precisa estar à mesma distância de **todos** os vértices; o da inscrita, à mesma distância de **todas** as faces. Um paralelepípedo retângulo sempre tem esfera circunscrita (o centro é o ponto de encontro das diagonais), mas só tem esfera inscrita se for um cubo: as faces opostas ficam a distâncias diferentes do centro.

Os **poliedros regulares** têm as três esferas, todas com o mesmo centro.

### Uma fórmula geral para o raio da esfera inscrita

Se um poliedro tem esfera inscrita de raio $r$ e centro $O$, ligue $O$ a cada face: o poliedro fica dividido em **pirâmides**, uma sobre cada face, todas com altura $r$ (o raio é perpendicular a cada face no ponto de tangência, notebook 13). Somando os volumes dessas pirâmides:
$$\text{volume} = \frac{1}{3} S_1 r + \frac{1}{3} S_2 r + \cdots = \frac{1}{3} \cdot S_{\text{total}} \cdot r \qquad\Longrightarrow\qquad \boxed{r = \frac{3 \cdot \text{volume}}{S_{\text{total}}}}$$

> 📎 **Conexão.** É a mesma ideia das "pirâmides com vértice no centro" que você usou no notebook 13 para encontrar a área da esfera.

🕹️ Teste a ideia num paralelepípedo: ele tem esfera circunscrita? E inscrita?

In [ ]:
def testar_paralelepipedo(comprimento: float, largura: float, altura: float) -> None:
    """Desenha um paralelepípedo retângulo com as esferas que ele tiver."""
    raio_circunscrita = math.sqrt(comprimento**2 + largura**2 + altura**2) / 2   # metade da diagonal
    distancias_as_faces = [comprimento / 2, largura / 2, altura / 2]
    tem_inscrita = (comprimento == largura == altura)

    # O paralelepípedo fica com o centro (o encontro das diagonais) na origem do desenho.
    caixa = u3d.paralelepipedo_retangulo(
        comprimento, largura, altura,
        origem=(-comprimento / 2, -largura / 2, -altura / 2),
    )
    fig = u3d.criar_figura(titulo="Paralelepípedo retângulo e suas esferas")
    fig = u3d.desenhar_poliedro(fig, *caixa, opacidade=0.3)
    fig = u3d.desenhar_esfera(
        fig, (0, 0, 0), raio_circunscrita, cor=u3d.CORES["plano_terciario"],
        opacidade=0.15,
    )
    if tem_inscrita:
        fig = u3d.desenhar_esfera(
            fig, (0, 0, 0), distancias_as_faces[0], cor=u3d.CORES["plano"],
            opacidade=0.4,
        )
    fig = u3d.desenhar_ponto(fig, (0, 0, 0), rotulo="O")
    u3d.mostrar(fig)

    print(f"Raio da esfera circunscrita (metade da diagonal): {raio_circunscrita:.4f}")
    print("Distâncias do centro às faces (metade de cada aresta): "
          + ", ".join(f"{d:g}" for d in distancias_as_faces))
    if tem_inscrita:
        print("As três distâncias são iguais: é um cubo, e ele TEM esfera inscrita.")
    else:
        print("As distâncias às faces não são todas iguais: NÃO há esfera inscrita.")

    # Conferência com o utils_3d: o maior e o menor valor de cada lista de distâncias.
    medidas = u3d.medir_distancias_ao_centro(*caixa)
    for chave in ["vertices", "faces"]:
        print(f"Conferência, distâncias aos {chave}: menor {min(medidas[chave]):.4f}, "
              f"maior {max(medidas[chave]):.4f}")


interact(
    testar_paralelepipedo,
    comprimento=widgets.FloatSlider(
        min=1, max=6, step=0.5, value=2, description="comprimento",
        continuous_update=False,
    ),
    largura=widgets.FloatSlider(
        min=1, max=6, step=0.5, value=4, description="largura",
        continuous_update=False,
    ),
    altura=widgets.FloatSlider(
        min=1, max=6, step=0.5, value=6, description="altura",
        continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=2.0, continuous_update=False, description='comprimento', max=6.0, min=…

**Pare e pense**

1. Uma pirâmide quadrangular regular sempre tem esfera inscrita? (Pense no centro: ele precisa estar no eixo da pirâmide, à mesma distância da base e das faces laterais.)
2. Um poliedro tem volume 60 e área total 45. Se ele tiver esfera inscrita, qual é o raio dela?

<details>
<summary>Ver respostas</summary>

1. Sim. Na seção meridiana que passa pelos apótemas, o centro é o ponto do eixo onde a bissetriz do ângulo entre a base e o apótema da pirâmide corta o eixo; por simetria, esse ponto está à mesma distância de todas as faces.
2. $r = \frac{3 \cdot 60}{45} = 4$.
</details>

## 2. Esfera e cubo

Num cubo de aresta $a$ e centro $O$:

| Esfera | Toca o cubo em | Raio | Por quê |
|---|---|---|---|
| inscrita | centros das 6 faces | $r = \dfrac{a}{2}$ | distância do centro a uma face: metade da aresta |
| tangente às arestas | pontos médios das 12 arestas | $\rho = \dfrac{a\sqrt{2}}{2}$ | metade da diagonal de uma face |
| circunscrita | os 8 vértices | $R = \dfrac{a\sqrt{3}}{2}$ | metade da diagonal do cubo (notebook 08) |

Repare: $r < \rho < R$, e as três razões são $1 : \sqrt{2} : \sqrt{3}$.

**Por que $\rho$ é metade da diagonal de uma face?** O centro do cubo e os pontos médios de duas arestas opostas (paralelas, em faces opostas) estão alinhados, e a distância entre esses pontos médios é a diagonal de uma face.

Nas figuras deste notebook, os pontos de tangência seguem sempre as mesmas cores: **vermelho** nas faces (esfera inscrita), **azul** nas arestas (esfera tangente às arestas) e **laranja** nos vértices (esfera circunscrita).

🕹️ Ligue e desligue as esferas.

In [4]:
# Duas funções de apoio, que vamos reaproveitar nas seções 3 e 4.
def pontos_de_tangencia(vertices: list, faces: list, centro: tuple) -> tuple:
    """Devolve os pontos onde as três esferas de centro `centro` tocam o poliedro."""
    # Esfera inscrita: o pé da perpendicular traçada do centro até cada face.
    nas_faces = [
        u3d.pe_da_perpendicular_ao_plano(
            centro, u3d.plano_por_tres_pontos(*[vertices[k] for k in face[:3]])
        )
        for face in faces
    ]
    # Esfera tangente às arestas: o pé da perpendicular do centro até cada aresta
    # (num poliedro regular, é o ponto médio da aresta).
    arestas = {
        (min(i, j), max(i, j))
        for face in faces
        for i, j in zip(face, face[1:] + face[:1])
    }
    nas_arestas = [
        u3d.pe_da_perpendicular_a_reta(centro, vertices[i], vertices[j])
        for i, j in sorted(arestas)
    ]
    # Esfera circunscrita: os próprios vértices.
    return nas_faces, nas_arestas, list(vertices)


def desenhar_esferas(
    fig, centro: tuple, raios: list, ligadas: list, pontos: tuple, mostrar_pontos: bool
):
    """Desenha as esferas ligadas (inscrita, tangente às arestas, circunscrita)."""
    cores_das_esferas = [
        u3d.CORES["plano"], u3d.CORES["plano_secundario"], u3d.CORES["plano_terciario"]
    ]
    cores_dos_pontos = [
        u3d.CORES["destaque"], u3d.CORES["reta"], u3d.CORES["reta_secundaria"]
    ]
    for raio, ligada, lista, cor_esfera, cor_pontos in zip(
        raios, ligadas, pontos, cores_das_esferas, cores_dos_pontos
    ):
        if ligada:
            fig = u3d.desenhar_esfera(fig, centro, raio, cor=cor_esfera, opacidade=0.25)
            if mostrar_pontos:
                fig = u3d.desenhar_pontos(fig, lista, cor=cor_pontos, tamanho=5)
    return fig


def explorar_cubo(
    aresta: float, inscrita: bool, tangente: bool, circunscrita: bool,
    mostrar_pontos: bool,
) -> None:
    """Desenha o cubo com as esferas escolhidas e calcula os três raios."""
    r = aresta / 2
    rho = aresta * math.sqrt(2) / 2
    R = aresta * math.sqrt(3) / 2

    vertices, faces = u3d.poliedro_de_platao("cubo", aresta)   # centro na origem
    centro = (0, 0, 0)
    pontos = pontos_de_tangencia(vertices, faces, centro)
    fig = u3d.criar_figura(titulo=f"Cubo de aresta {aresta:g} e suas esferas")
    fig = u3d.desenhar_poliedro(fig, vertices, faces, opacidade=0.25)
    fig = desenhar_esferas(
        fig, centro, [r, rho, R], [inscrita, tangente, circunscrita], pontos,
        mostrar_pontos,
    )
    u3d.mostrar(fig)

    print(f"Esfera inscrita: r = a/2 = {r:.4f}")
    print(f"Esfera tangente às arestas: ρ = a√2/2 = {rho:.4f}")
    print(f"Esfera circunscrita: R = a√3/2 = {R:.4f}")
    medidos = u3d.raios_do_poliedro_regular(vertices, faces)
    print(f"Conferência com o utils_3d: r = {medidos['inscrita']:.4f}, "
          f"ρ = {medidos['arestas']:.4f}, R = {medidos['circunscrita']:.4f}")


interact(
    explorar_cubo,
    aresta=widgets.FloatSlider(
        min=1, max=4, step=0.5, value=2, description="aresta a",
        continuous_update=False,
    ),
    inscrita=widgets.Checkbox(value=True, description="Inscrita"),
    tangente=widgets.Checkbox(value=False, description="Tangente às arestas"),
    circunscrita=widgets.Checkbox(value=False, description="Circunscrita"),
    mostrar_pontos=widgets.Checkbox(value=True, description="Mostrar pontos de tangência"),
);

interactive(children=(FloatSlider(value=2.0, continuous_update=False, description='aresta a', max=4.0, min=1.0…

**Pare e pense**

1. Num cubo de aresta 6, quanto medem $r$, $\rho$ e $R$?
2. Uma esfera de raio 5 está circunscrita a um cubo. Qual é a aresta do cubo?

<details>
<summary>Ver respostas</summary>

1. $r = 3$; $\rho = 3\sqrt{2} \approx 4{,}24$; $R = 3\sqrt{3} \approx 5{,}20$.
2. $\frac{a\sqrt{3}}{2} = 5$, então $a = \frac{10}{\sqrt{3}} = \frac{10\sqrt{3}}{3} \approx 5{,}77$.
</details>

## 3. Esfera e octaedro regular

O octaedro regular de aresta $a$ pode ser visto como **duas pirâmides quadrangulares regulares** coladas pela base: a base comum é um quadrado de lado $a$, e o centro $O$ do octaedro é o centro desse quadrado.

**Esfera circunscrita.** Os 6 vértices estão à mesma distância de $O$: 4 deles são vértices do quadrado (metade da diagonal: $\frac{a\sqrt{2}}{2}$), e os outros 2 são os vértices das pirâmides, a uma altura que, pelo triângulo $VOA$ do notebook 10 ($a^2 = h^2 + R^2$ com $R = \frac{a\sqrt{2}}{2}$), também vale $\frac{a\sqrt{2}}{2}$:
$$R = \frac{a\sqrt{2}}{2}$$

**Esfera tangente às arestas.** A distância de $O$ ao ponto médio de um lado do quadrado é metade do lado:
$$\rho = \frac{a}{2}$$

**Esfera inscrita.** Pela fórmula geral: o volume do octaedro é o de duas pirâmides, $2 \cdot \frac{1}{3} a^2 \cdot \frac{a\sqrt{2}}{2} = \frac{a^3\sqrt{2}}{3}$, e a área é a de 8 triângulos equiláteros, $2a^2\sqrt{3}$. Então:
$$r = \frac{3 \cdot \frac{a^3\sqrt{2}}{3}}{2a^2\sqrt{3}} = \frac{a\sqrt{2}}{2\sqrt{3}} = \frac{a\sqrt{6}}{6}$$

<details>
<summary>Outra forma: o triângulo VOM</summary>

Na pirâmide de cima (vértice $V$, altura $h = \frac{a\sqrt{2}}{2}$, apótema da base $m = \frac{a}{2}$, apótema $g = \frac{a\sqrt{3}}{2}$), o raio $r$ é a altura do triângulo retângulo $VOM$ relativa à hipotenusa $VM$. Pela área do triângulo: $r \cdot g = h \cdot m$, então $r = \frac{h \cdot m}{g} = \frac{a\sqrt{6}}{6}$.
</details>

🕹️ Explore o octaedro e suas esferas.

In [ ]:
def explorar_octaedro(
    aresta: float, inscrita: bool, tangente: bool, circunscrita: bool,
    mostrar_pontos: bool, separar_piramides: bool,
) -> None:
    """Desenha o octaedro com as esferas escolhidas e calcula os raios."""
    R = aresta * math.sqrt(2) / 2
    rho = aresta / 2
    volume = aresta**3 * math.sqrt(2) / 3
    area = 2 * aresta**2 * math.sqrt(3)
    r = 3 * volume / area

    vertices, faces = u3d.poliedro_de_platao("octaedro", aresta)   # centro na origem
    centro = (0, 0, 0)
    fig = u3d.criar_figura(titulo=f"Octaedro regular de aresta {aresta:g}")
    if separar_piramides:
        # Os vértices 0 a 3 formam o quadrado do "meio" (nesta ordem, em volta dele);
        # os vértices 4 e 5 são as pontas de cima e de baixo.
        quadrado = [vertices[k] for k in (0, 2, 1, 3)]
        de_cima = u3d.piramide(quadrado, vertices[4])
        de_baixo = u3d.piramide(quadrado, vertices[5])
        de_cima, de_baixo = u3d.afastar_pecas([de_cima, de_baixo], aresta / 2)
        fig = u3d.desenhar_poliedro(fig, *de_cima, cor=u3d.CORES["plano"], opacidade=0.5)
        fig = u3d.desenhar_poliedro(
            fig, *de_baixo, cor=u3d.CORES["plano_secundario"], opacidade=0.5
        )
    else:
        pontos = pontos_de_tangencia(vertices, faces, centro)
        fig = u3d.desenhar_poliedro(fig, vertices, faces, opacidade=0.25)
        fig = desenhar_esferas(
            fig, centro, [r, rho, R], [inscrita, tangente, circunscrita], pontos,
            mostrar_pontos,
        )
    u3d.mostrar(fig)

    if separar_piramides:
        print("(As esferas ficam escondidas enquanto as pirâmides estão separadas.)")
    print(f"Esfera circunscrita: R = a√2/2 = {R:.4f}")
    print(f"Esfera tangente às arestas: ρ = a/2 = {rho:.4f}")
    print(f"Volume = a³√2/3 = {volume:.4f}")
    print(f"Área total = 2a²√3 = {area:.4f}")
    print(f"Esfera inscrita: r = 3 · volume / área = {r:.4f}")
    medidos = u3d.raios_do_poliedro_regular(vertices, faces)
    print(f"Conferência com o utils_3d: r = {medidos['inscrita']:.4f}, "
          f"ρ = {medidos['arestas']:.4f}, R = {medidos['circunscrita']:.4f}, "
          f"volume = {u3d.medir_volume(vertices, faces):.4f}, "
          f"área = {u3d.medir_area_total(vertices, faces):.4f}")


interact(
    explorar_octaedro,
    aresta=widgets.FloatSlider(
        min=1, max=4, step=0.5, value=2, description="aresta a",
        continuous_update=False,
    ),
    inscrita=widgets.Checkbox(value=True, description="Inscrita"),
    tangente=widgets.Checkbox(value=False, description="Tangente às arestas"),
    circunscrita=widgets.Checkbox(value=False, description="Circunscrita"),
    mostrar_pontos=widgets.Checkbox(value=True, description="Mostrar pontos de tangência"),
    separar_piramides=widgets.Checkbox(
        value=False, description="Mostrar a divisão em duas pirâmides"
    ),
);

interactive(children=(FloatSlider(value=2.0, continuous_update=False, description='aresta a', max=4.0, min=1.0…

**Pare e pense**

1. Num octaedro regular de aresta 4, quanto medem $R$ e $r$?
2. O octaedro regular e o cubo têm a mesma razão $\frac{R}{r}$. Qual é ela?

<details>
<summary>Ver respostas</summary>

1. $R = 2\sqrt{2} \approx 2{,}83$; $r = \frac{4\sqrt{6}}{6} \approx 1{,}63$.
2. $\sqrt{3}$. No cubo: $\frac{a\sqrt{3}/2}{a/2} = \sqrt{3}$. No octaedro: $\frac{a\sqrt{2}/2}{a\sqrt{6}/6} = \frac{3\sqrt{2}}{\sqrt{6}} = \sqrt{3}$. (Na seção 5 você vai descobrir por que isso não é coincidência.)
</details>

## 4. Esfera e tetraedro regular

No tetraedro regular de aresta $a$, a altura é $h = \frac{a\sqrt{6}}{3}$ (notebook 10).

**Esfera inscrita.** Pela fórmula geral: o tetraedro tem 4 faces iguais, de área $S_f$, e volume $\frac{1}{3} S_f h$. Então:
$$r = \frac{3 \cdot \frac{1}{3} S_f h}{4 S_f} = \frac{h}{4} \qquad\Longrightarrow\qquad \boxed{r = \frac{h}{4} = \frac{a\sqrt{6}}{12}}$$

**Esfera circunscrita.** O centro $O$ está na altura do tetraedro, a uma distância $r = \frac{h}{4}$ da base. Então a distância de $O$ ao vértice de cima é o resto da altura:
$$\boxed{R = h - r = \frac{3h}{4} = \frac{a\sqrt{6}}{4}}$$

O centro divide a altura na razão $1 : 3$ (um quarto embaixo, três quartos em cima). Repare que $\frac{R}{r} = 3$.

**Esfera tangente às arestas:** $\rho = \dfrac{a\sqrt{2}}{4}$.

### O tetraedro dentro do cubo

Escolha quatro vértices de um cubo que não são vizinhos dois a dois, como $A$, $C$, $F$ e $H$. Eles formam um **tetraedro regular**: cada aresta do tetraedro é uma diagonal de face do cubo, de medida $a\sqrt{2}$. Isso dá três fatos interessantes:
- o tetraedro e o cubo têm a **mesma esfera circunscrita**;
- o volume do tetraedro é **um terço** do volume do cubo (o cubo menos 4 "cantos", que são pirâmides de volume $\frac{a^3}{6}$ cada: $a^3 - 4 \cdot \frac{a^3}{6} = \frac{a^3}{3}$);
- a esfera tangente às arestas do tetraedro é a esfera **inscrita** no cubo (os pontos médios das arestas do tetraedro são os centros das faces do cubo).

> 🎯 **Aplicação: a molécula de metano.** No metano (CH₄), o átomo de carbono fica no centro de um tetraedro regular, e os quatro átomos de hidrogênio, nos vértices. O ângulo entre duas ligações é o ângulo $A\hat{O}C$, entre dois raios da esfera circunscrita. Você vai calculá-lo no Desafio 7: dá cerca de $109{,}5°$, um número famoso na Química.

🕹️ Explore o tetraedro, sozinho ou dentro do cubo.

In [ ]:
def explorar_tetraedro(
    vista: str, aresta: float, inscrita: bool, tangente: bool, circunscrita: bool,
    mostrar_pontos: bool,
) -> None:
    """Desenha o tetraedro regular com suas esferas, ou dentro de um cubo."""
    if vista == "Tetraedro e suas esferas":
        h = aresta * math.sqrt(6) / 3
        r = h / 4
        R = 3 * h / 4
        rho = aresta * math.sqrt(2) / 4

        # O tetraedro regular é uma pirâmide triangular regular de altura h, com a
        # base deitada e o centro da base no ponto (0, 0, 0) do desenho.
        vertices, faces = u3d.piramide_regular(3, aresta, h)
        centro = (0, 0, r)             # O fica na altura, a h/4 da base
        topo, pe_da_altura = vertices[3], (0, 0, 0)
        pontos = pontos_de_tangencia(vertices, faces, centro)

        fig = u3d.criar_figura(titulo=f"Tetraedro regular de aresta {aresta:g}")
        fig = u3d.desenhar_poliedro(fig, vertices, faces, opacidade=0.25)
        fig = desenhar_esferas(
            fig, centro, [r, rho, R], [inscrita, tangente, circunscrita], pontos,
            mostrar_pontos,
        )
        # A altura do tetraedro, com o centro O dividindo-a em h/4 e 3h/4.
        fig = u3d.desenhar_segmento(
            fig, topo, pe_da_altura, cor=u3d.CORES["apoio"], tracejado=True
        )
        fig = u3d.desenhar_ponto(fig, centro, rotulo="O")
        folga = 0.15 * aresta   # só para os rótulos não ficarem em cima do segmento
        fig = u3d.adicionar_rotulo(fig, (folga, 0, r / 2), "h/4")
        fig = u3d.adicionar_rotulo(fig, (folga, 0, r + R / 2), "3h/4")
        u3d.mostrar(fig)

        print(f"Altura: h = a√6/3 = {h:.4f}")
        print(f"Esfera inscrita: r = h/4 = {r:.4f}")
        print(f"Esfera circunscrita: R = 3h/4 = {R:.4f}")
        print(f"Esfera tangente às arestas: ρ = a√2/4 = {rho:.4f}")
        medidos = u3d.raios_do_poliedro_regular(vertices, faces)
        print(f"Conferência com o utils_3d: r = {medidos['inscrita']:.4f}, "
              f"ρ = {medidos['arestas']:.4f}, R = {medidos['circunscrita']:.4f}")
        return

    # Segunda vista: aqui `aresta` é a aresta do CUBO.
    aresta_do_tetraedro = aresta * math.sqrt(2)
    volume_cubo = aresta**3
    volume_tetraedro = volume_cubo - 4 * (aresta**3 / 6)
    raio_circunscrita = aresta * math.sqrt(3) / 2     # metade da diagonal do cubo

    metade = aresta / 2
    v = u3d.vertices_do_cubo(aresta, origem=(-metade, -metade, -metade))
    tetraedro = u3d.piramide([v["A"], v["C"], v["F"]], v["H"])
    fig = u3d.criar_figura(titulo="O tetraedro ACFH dentro do cubo")
    fig = u3d.desenhar_cubo(fig, aresta, origem=(-metade, -metade, -metade), opacidade=0.15)
    fig = u3d.desenhar_poliedro(fig, *tetraedro, cor=u3d.CORES["plano"], opacidade=0.5)
    fig = u3d.desenhar_esfera(
        fig, (0, 0, 0), raio_circunscrita, cor=u3d.CORES["plano_terciario"],
        opacidade=0.1,
    )
    if mostrar_pontos:
        # Os pontos médios das arestas do tetraedro: os centros das faces do cubo.
        _, meios_das_arestas, _ = pontos_de_tangencia(*tetraedro, (0, 0, 0))
        fig = u3d.desenhar_pontos(fig, meios_das_arestas, cor=u3d.CORES["reta"])
    u3d.mostrar(fig)

    print(f"Aresta do tetraedro (diagonal de uma face do cubo): a√2 = {aresta_do_tetraedro:.4f}")
    print(f"Volume do cubo: a³ = {volume_cubo:.4f}")
    print(f"Volume do tetraedro: a³ − 4 · a³/6 = {volume_tetraedro:.4f} "
          f"(um terço do cubo: {volume_cubo / 3:.4f})")
    print(f"Raio da esfera circunscrita comum: a√3/2 = {raio_circunscrita:.4f}")
    medidos = u3d.raios_do_poliedro_regular(*tetraedro)
    print(f"Conferência com o utils_3d: volume do tetraedro = "
          f"{u3d.medir_volume(*tetraedro):.4f}, R = {medidos['circunscrita']:.4f}, "
          f"ρ do tetraedro = {medidos['arestas']:.4f} (= a/2, o raio inscrito do cubo)")


interact(
    explorar_tetraedro,
    vista=widgets.RadioButtons(
        options=["Tetraedro e suas esferas", "Tetraedro dentro do cubo"],
        value="Tetraedro e suas esferas",
        layout=widgets.Layout(width="max-content"),
    ),
    aresta=widgets.FloatSlider(
        min=1, max=4, step=0.5, value=2, description="aresta a",
        continuous_update=False,
    ),
    inscrita=widgets.Checkbox(value=True, description="Inscrita"),
    tangente=widgets.Checkbox(value=False, description="Tangente às arestas"),
    circunscrita=widgets.Checkbox(value=False, description="Circunscrita"),
    mostrar_pontos=widgets.Checkbox(value=True, description="Mostrar pontos de tangência"),
);

interactive(children=(RadioButtons(description='vista', layout=Layout(width='max-content'), options=('Tetraedr…

**Pare e pense**

1. Num tetraedro regular de aresta 6, quanto medem $h$, $r$ e $R$?
2. Um cubo tem aresta 3. Qual é a aresta e qual é o volume do tetraedro regular formado por quatro dos seus vértices?

<details>
<summary>Ver respostas</summary>

1. $h = 2\sqrt{6} \approx 4{,}90$; $r = \frac{\sqrt{6}}{2} \approx 1{,}22$; $R = \frac{3\sqrt{6}}{2} \approx 3{,}67$.
2. Aresta $3\sqrt{2} \approx 4{,}24$; volume $\frac{27}{3} = 9$.
</details>

## 5. Os cinco poliedros regulares e suas esferas

Todo poliedro regular tem as três esferas, com o mesmo centro. A tabela abaixo mostra os raios para **aresta 1**. Para outra aresta $a$, basta multiplicar por $a$ (semelhança, notebook 14).

As fórmulas do dodecaedro e do icosaedro envolvem o número $\sqrt{5}$ e ficam fora do nosso escopo; os valores foram medidos com Python.

**Duais trocam as esferas.** No notebook 07, você viu que o dual de um poliedro tem um vértice no centro de cada face. Então, se o dual for construído assim, a esfera **inscrita** no poliedro original passa pelos vértices do dual: vira a esfera **circunscrita** ao dual. Uma consequência: poliedros duais têm a **mesma razão** $\frac{R}{r}$. Cubo e octaedro: $\sqrt{3}$. Dodecaedro e icosaedro: cerca de $1{,}258$. O tetraedro, dual de si mesmo, tem a maior razão: $3$.

> 🕵️ **Curiosidade histórica.** Em 1596, **Johannes Kepler** publicou *Mysterium Cosmographicum* ("O mistério cósmico"). Na época, conheciam-se seis planetas, e Kepler achava que as distâncias entre as suas órbitas eram determinadas pelos cinco poliedros regulares. A ideia: a órbita de Saturno ficaria numa esfera; dentro dela, um **cubo** inscrito; dentro do cubo, a esfera inscrita, de Júpiter; dentro dela, um **tetraedro**; depois Marte, um **dodecaedro**, a Terra, um **icosaedro**, Vênus, um **octaedro** e, por fim, Mercúrio. Cada poliedro inscrito numa esfera e circunscrito à seguinte. A teoria estava errada, mas o esforço de Kepler para ajustá-la aos dados o levou, anos depois, às leis corretas do movimento dos planetas.

Monte a tabela e o "universo" de Kepler.

In [7]:
solidos = ["tetraedro", "cubo", "octaedro", "dodecaedro", "icosaedro"]

print(f"{'poliedro':<12}{'R':>9}{'ρ':>9}{'r':>9}{'R/r':>9}")
for nome in solidos:
    vertices, faces = u3d.poliedro_de_platao(nome, aresta=1)
    raios = u3d.raios_do_poliedro_regular(vertices, faces)
    R, rho, r = raios["circunscrita"], raios["arestas"], raios["inscrita"]
    razao = R / r
    print(f"{nome:<12}{R:>9.4f}{rho:>9.4f}{r:>9.4f}{razao:>9.4f}")

# A fórmula geral r = 3 · volume / S vale para os cinco.
print()
print(f"{'poliedro':<12}{'3·volume/S':>12}{'r medido':>11}")
for nome in solidos:
    vertices, faces = u3d.poliedro_de_platao(nome, aresta=1)
    volume = u3d.medir_volume(vertices, faces)
    area = u3d.medir_area_total(vertices, faces)
    r_pela_formula = 3 * volume / area
    r_medido = u3d.raios_do_poliedro_regular(vertices, faces)["inscrita"]
    print(f"{nome:<12}{r_pela_formula:>12.4f}{r_medido:>11.4f}")

poliedro            R        ρ        r      R/r
tetraedro      0.6124   0.3536   0.2041   3.0000
cubo           0.8660   0.7071   0.5000   1.7321
octaedro       0.7071   0.5000   0.4082   1.7321
dodecaedro     1.4013   1.3090   1.1135   1.2584
icosaedro      0.9511   0.8090   0.7558   1.2584

poliedro      3·volume/S   r medido
tetraedro         0.2041     0.2041
cubo              0.5000     0.5000
octaedro          0.4082     0.4082
dodecaedro        1.1135     1.1135
icosaedro         0.7558     0.7558


In [8]:
ordem = ["cubo", "tetraedro", "dodecaedro", "icosaedro", "octaedro"]
planetas = ["Saturno", "Júpiter", "Marte", "Terra", "Vênus", "Mercúrio"]
raio_atual = 10
raios_das_esferas = [raio_atual]
fatores = []
for nome in ordem:
    vertices, faces = u3d.poliedro_de_platao(nome, aresta=1)
    raios = u3d.raios_do_poliedro_regular(vertices, faces)
    fator = raio_atual / raios["circunscrita"]          # ampliar até tocar a esfera de fora
    raio_atual = raios["inscrita"] * fator               # a esfera de dentro
    raios_das_esferas.append(raio_atual)
    fatores.append(fator)

cores_kepler = [
    u3d.CORES["plano"], u3d.CORES["plano_secundario"], u3d.CORES["plano_terciario"],
    u3d.CORES["destaque"], u3d.CORES["reta_secundaria"],
]


def montar_universo(camadas: int, mostrar_esferas: bool) -> None:
    """Desenha os `camadas` primeiros poliedros do modelo de Kepler, de fora para dentro."""
    fig = u3d.criar_figura(titulo="O universo de Kepler", altura=600)
    for k in range(camadas):
        # O k-ésimo poliedro, ampliado até ficar inscrito na esfera k.
        vertices, faces = u3d.poliedro_de_platao(ordem[k], aresta=1)
        vertices, faces = u3d.escalar_poliedro(vertices, faces, fatores[k])
        fig = u3d.desenhar_poliedro(fig, vertices, faces, cor=cores_kepler[k], opacidade=0.2)
    if mostrar_esferas:
        # Resolução baixa: são até seis esferas na mesma figura.
        for k in range(camadas + 1):
            fig = u3d.desenhar_esfera(
                fig, (0, 0, 0), raios_das_esferas[k], cor=u3d.CORES["apoio"],
                opacidade=0.08, resolucao=25,
            )
    u3d.mostrar(fig)

    print(f"{'planeta':<10}{'raio da esfera':>15}   poliedro inscrito nela")
    for k in range(camadas + 1):
        poliedro = ordem[k] if k < camadas else "—"
        print(f"{planetas[k]:<10}{raios_das_esferas[k]:>15.4f}   {poliedro}")
    razao = raios_das_esferas[0] / raios_das_esferas[camadas]
    print(f"Razão entre a esfera de Saturno e a de {planetas[camadas]}: {razao:.4f}")


interact(
    montar_universo,
    camadas=widgets.IntSlider(
        min=1, max=5, step=1, value=5, description="camadas",
        continuous_update=False,
    ),
    mostrar_esferas=widgets.Checkbox(value=True, description="Mostrar esferas"),
);

interactive(children=(IntSlider(value=5, continuous_update=False, description='camadas', max=5, min=1), Checkb…

**Pare e pense**

1. Qual dos cinco poliedros regulares tem a esfera inscrita "mais folgada" dentro da circunscrita (maior $\frac{R}{r}$)? E a mais "justa"?
2. Por que o octaedro e o cubo têm a mesma razão $\frac{R}{r}$?

<details>
<summary>Ver respostas</summary>

1. Mais folgada: o tetraedro ($\frac{R}{r} = 3$). Mais justa: o dodecaedro e o icosaedro (cerca de 1,258). Quanto mais faces, mais o poliedro "se parece" com uma esfera.
2. Porque são duais: a esfera inscrita de um corresponde à circunscrita do outro, e as razões se mantêm.
</details>

## Resumo

**Esfera inscrita** (tangente às faces, raio $r$) · **tangente às arestas** (raio $\rho$) · **circunscrita** (passa pelos vértices, raio $R$). "Inscrito" é sempre quem fica dentro.

**Fórmula geral:** se existe esfera inscrita, $r = \dfrac{3 \cdot \text{volume}}{S_{\text{total}}}$.

| Poliedro (aresta $a$) | $r$ | $\rho$ | $R$ | $R/r$ |
|---|---|---|---|---|
| cubo | $\dfrac{a}{2}$ | $\dfrac{a\sqrt{2}}{2}$ | $\dfrac{a\sqrt{3}}{2}$ | $\sqrt{3}$ |
| octaedro | $\dfrac{a\sqrt{6}}{6}$ | $\dfrac{a}{2}$ | $\dfrac{a\sqrt{2}}{2}$ | $\sqrt{3}$ |
| tetraedro | $\dfrac{a\sqrt{6}}{12} = \dfrac{h}{4}$ | $\dfrac{a\sqrt{2}}{4}$ | $\dfrac{a\sqrt{6}}{4} = \dfrac{3h}{4}$ | $3$ |

Dodecaedro e icosaedro: $\dfrac{R}{r} \approx 1{,}258$.

**Tetraedro no cubo:** aresta $a\sqrt{2}$, volume $\dfrac{a^3}{3}$, mesma esfera circunscrita.

**Duais** trocam esfera inscrita e circunscrita e têm a mesma razão $\dfrac{R}{r}$.

**Embalagens:** esfera inscrita no cubo ⟹ cerca de 48% de ar; cubo inscrito na esfera ⟹ cerca de 63% de ar.

## Exercícios

Responda preenchendo as variáveis indicadas em cada célula (troque cada `...` pela sua resposta) e rode a célula de verificação logo abaixo. Os exercícios estão em três níveis: **básico**, **intermediário** e **desafio**. Use `math.pi` e `math.sqrt` nos cálculos.

O gabarito comentado está em [`0615b_inscricao_e_circunscricao_esfera_e_poliedros_gabarito.ipynb`](0615b_inscricao_e_circunscricao_esfera_e_poliedros_gabarito.ipynb). Tente sozinho antes de olhar!

### Básico

In [ ]:
# Exercício Básico 1: verdadeiro ou falso? Troque cada ... por True ou False.
# (a) Todo poliedro tem esfera inscrita.
# (b) O raio da esfera inscrita num cubo é metade da aresta.
# (c) O diâmetro da esfera circunscrita a um cubo é igual à diagonal do cubo.
# (d) No tetraedro regular, o raio da esfera inscrita é um quarto da altura.
# (e) Um paralelepípedo retângulo que não é cubo tem esfera circunscrita.
respostas_ex1 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
    "e": ...,
}

print(respostas_ex1)

In [ ]:
# Verificação
gabarito_ex1 = {"a": False, "b": True, "c": True, "d": True, "e": True}
dicas_ex1 = {
    "a": "um paralelepípedo que não é cubo não tem esfera inscrita (seção 1 - Esferas inscrita, circunscrita e tangente às arestas)",
    "b": "a distância do centro do cubo a uma face é metade da aresta (seção 2 - Esfera e cubo)",
    "c": "R é metade da diagonal do cubo (seção 2 - Esfera e cubo)",
    "d": "pela fórmula r = 3 · volume / S, r = h/4 (seção 4 - Esfera e tetraedro regular)",
    "e": "o centro é o encontro das diagonais, à mesma distância dos 8 vértices (seção 1 - Esferas inscrita, circunscrita e tangente às arestas)",
}
for item, certa in gabarito_ex1.items():
    assert respostas_ex1[item] == certa, (
        f"Tente de novo, revise: o item ({item}) não está certo; {dicas_ex1[item]}."
    )
print("Certo!")

In [ ]:
# Exercício Básico 2: o cubo. Num cubo de aresta 6, calcule:
# (a) o raio r da esfera inscrita;
# (b) o raio ρ da esfera tangente às arestas;
# (c) o raio R da esfera circunscrita.
r_ex2 = ...
rho_ex2 = ...
R_ex2 = ...

print(r_ex2, rho_ex2, R_ex2)

In [ ]:
# Verificação
respostas_ex2 = [
    ("a", r_ex2, 3),
    ("b", rho_ex2, 3 * math.sqrt(2)),
    ("c", R_ex2, 3 * math.sqrt(3)),
]
for item, resposta, certa in respostas_ex2:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), r = a/2, ρ = metade da diagonal da "
        "face, R = metade da diagonal do cubo (seção 2 - Esfera e cubo)."
    )
print("Certo!")

In [ ]:
# Exercício Básico 3: o tetraedro. Num tetraedro regular de aresta 6, calcule:
# (a) a altura h;
# (b) o raio r da esfera inscrita;
# (c) o raio R da esfera circunscrita.
altura_ex3 = ...
r_ex3 = ...
R_ex3 = ...

print(altura_ex3, r_ex3, R_ex3)

In [ ]:
# Verificação
respostas_ex3 = [
    ("a", altura_ex3, 2 * math.sqrt(6)),
    ("b", r_ex3, math.sqrt(6) / 2),
    ("c", R_ex3, 3 * math.sqrt(6) / 2),
]
for item, resposta, certa in respostas_ex3:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), h = a√6/3, r = h/4, R = 3h/4 "
        "(seção 4 - Esfera e tetraedro regular)."
    )
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 4: a bola na caixa. Uma bola está inscrita numa caixa
# cúbica de aresta 20 cm. Calcule:
# (a) o volume de ar dentro da caixa, em cm³;
# (b) a porcentagem do espaço da caixa ocupada por ar.
ar_ex4 = ...
porcentagem_ex4 = ...

print(ar_ex4, porcentagem_ex4)

In [ ]:
# Verificação
ar_certo = 8000 - 4000 * math.pi / 3
respostas_ex4 = [("a", ar_ex4, ar_certo), ("b", porcentagem_ex4, ar_certo / 8000 * 100)]
for item, resposta, certa in respostas_ex4:
    assert resposta is not ... and abs(resposta - certa) <= 0.1, (
        f"Tente de novo, revise: no item ({item}), o raio da bola é metade da aresta; "
        "a porcentagem é ar / volume da caixa × 100 (seção 2 - Esfera e cubo)."
    )
print("Certo!")

In [ ]:
# Exercício Intermediário 5: o octaedro. Para um octaedro regular de aresta 4,
# calcule:
# (a) o raio R da esfera circunscrita;
# (b) o volume;
# (c) a área total;
# (d) o raio r da esfera inscrita, usando r = 3 · volume / S.
R_ex5 = ...
volume_ex5 = ...
area_ex5 = ...
r_ex5 = ...

print(R_ex5, volume_ex5, area_ex5, r_ex5)

In [ ]:
# Verificação
respostas_ex5 = [
    ("a", R_ex5, 2 * math.sqrt(2), "R = a√2/2"),
    ("b", volume_ex5, 64 * math.sqrt(2) / 3, "volume = a³√2/3"),
    ("c", area_ex5, 32 * math.sqrt(3), "S = 2a²√3"),
    ("d", r_ex5, 2 * math.sqrt(6) / 3, "r = 3 · volume / S"),
]
for item, resposta, certa, dica in respostas_ex5:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), {dica} (seção 3 - Esfera e octaedro regular)."
    )
print("Certo!")

In [ ]:
# Exercício Intermediário 6: o tetraedro no cubo. Quatro vértices de um cubo de
# aresta 3 formam um tetraedro regular. Calcule:
# (a) a aresta do tetraedro;
# (b) o volume do tetraedro.
aresta_ex6 = ...
volume_ex6 = ...

print(aresta_ex6, volume_ex6)

In [ ]:
# Verificação
respostas_ex6 = [("a", aresta_ex6, 3 * math.sqrt(2)), ("b", volume_ex6, 9)]
for item, resposta, certa in respostas_ex6:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), a aresta do tetraedro é a diagonal "
        "de uma face do cubo; o volume é um terço do volume do cubo (seção 4 - Esfera e tetraedro regular)."
    )
print("Certo!")

### Desafio

In [ ]:
# Exercício Desafio 7: o ângulo do metano. O carbono do metano fica no centro de um
# tetraedro regular, e os hidrogênios, nos vértices. Calcule, em graus, o ângulo
# entre duas ligações.
# Dica: coloque o tetraedro dentro de um cubo de aresta 2, com vértices em
# A(0, 0, 0), C(2, 2, 0), F(2, 0, 2) e H(0, 2, 2); o centro é O(1, 1, 1). No triângulo
# AOC, OA e OC são metades de diagonais do cubo, e AC é a diagonal de uma face. Use a
# lei dos cossenos (e math.acos e math.degrees).
angulo_ex7 = ...

print(angulo_ex7)

In [ ]:
# Verificação
assert angulo_ex7 is not ... and abs(angulo_ex7 - math.degrees(math.acos(-1 / 3))) <= 0.1, (
    "Tente de novo, revise: OA = OC = √3 e AC = 2√2; "
    "cos θ = (OA² + OC² − AC²) / (2 · OA · OC) (seção 4 - Esfera e tetraedro regular)."
)

# Conferência com o utils_3d e o desenho da molécula: o cubo, o tetraedro e as
# quatro ligações saindo do carbono, no centro.
v = u3d.vertices_do_cubo(2)
carbono = (1, 1, 1)
print(f"Ângulo medido com o utils_3d: {u3d.medir_angulo(carbono, v['A'], v['C']):.4f}°")
fig = u3d.criar_figura(titulo="A molécula de metano dentro do cubo")
fig = u3d.desenhar_cubo(fig, 2, opacidade=0.05)
fig = u3d.desenhar_poliedro(
    fig, *u3d.piramide([v["A"], v["C"], v["F"]], v["H"]), opacidade=0.15
)
for nome in "ACFH":
    fig = u3d.desenhar_segmento(fig, carbono, v[nome], cor=u3d.CORES["destaque"], espessura=6)
fig = u3d.desenhar_ponto(fig, carbono, rotulo="C (carbono)", tamanho=10)
u3d.mostrar(fig)
print("Certo!")

In [ ]:
# Exercício Desafio 8: o universo de Kepler. Usando u3d.raios_do_poliedro_regular,
# calcule a razão entre o raio da esfera de Saturno e o raio da esfera de Mercúrio
# no modelo de Kepler: o produto das razões R/r do cubo, do tetraedro, do
# dodecaedro, do icosaedro e do octaedro. Complete o laço.
ordem_ex8 = ["cubo", "tetraedro", "dodecaedro", "icosaedro", "octaedro"]
razao_ex8 = ...  # complete aqui: qual é o valor inicial de um produto?
for nome in ordem_ex8:
    vertices, faces = u3d.poliedro_de_platao(nome, aresta=1)
    raios = u3d.raios_do_poliedro_regular(vertices, faces)
    ...  # complete aqui: multiplique razao_ex8 pela razão R/r deste poliedro

print(razao_ex8)

In [ ]:
# Verificação
assert razao_ex8 is not ... and abs(razao_ex8 - 14.25) <= 0.05, (
    "Tente de novo, revise: multiplique, num laço, as razões R/r dos cinco "
    "poliedros, na ordem de Kepler, começando o produto em 1 (seção 5 - Os cinco poliedros regulares e suas esferas)."
)
print("Certo!")

## Conexões

**Isso depende de:**
- [`0603a_perpendicularidade.ipynb`](0603a_perpendicularidade.ipynb) e [`0604a_projecoes_distancias_e_angulos.ipynb`](0604a_projecoes_distancias_e_angulos.ipynb) (o pé da perpendicular e as distâncias de ponto a plano e de ponto a reta, que localizam os pontos de tangência)
- [`0607a_poliedros_convexos.ipynb`](0607a_poliedros_convexos.ipynb) (poliedros regulares e duais)
- [`0608a_prismas_e_paralelepipedos.ipynb`](0608a_prismas_e_paralelepipedos.ipynb) (diagonais do cubo)
- [`0610a_piramide.ipynb`](0610a_piramide.ipynb) (relações métricas e tetraedro regular; as pirâmides de mesmo vértice dentro do cubo)
- [`0613a_esfera.ipynb`](0613a_esfera.ipynb) (plano tangente, volume e a ideia das pirâmides com vértice no centro)
- [`0614a_solidos_semelhantes_e_troncos.ipynb`](0614a_solidos_semelhantes_e_troncos.ipynb) (raios proporcionais à aresta)

**Isso será usado em:**
- [`0616a_inscricao_e_circunscricao_corpos_redondos.ipynb`](0616a_inscricao_e_circunscricao_corpos_redondos.ipynb) (esfera, cilindro e cone combinados)
- 📌 Geometria Analítica (centro e raio de esferas por coordenadas)